## Gold Layer — Business Logic

Builds the gold layer from `workspace.silver.job_postings_skills` into `workspace.gold`:

* `gold.job_postings` — fact table with business logic: salary bands, seniority levels, remote flag
* `gold.skill_demand` — skill demand aggregate (job counts, % of postings, salary stats)
* `gold.location_summary` — postings and salaries by city/region

Each daily load keeps its own `ingestion_date` — new days are added alongside old ones, never replacing them. Salaries are in US dollars (`avg_salary_usd` etc.). Run top-to-bottom after the Bronze to Silver pipeline.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.job_postings AS
SELECT
  'US' AS country,
  job_id, title, company, location_city, location_region, category, contract_time,
  salary_min, salary_max, salary_mid,
  salary_currency,
  CASE
    WHEN salary_mid IS NULL THEN 'unknown'
    WHEN salary_mid < 70000 THEN 'under 70k'
    WHEN salary_mid < 100000 THEN '70k - 100k'
    WHEN salary_mid < 130000 THEN '100k - 130k'
    ELSE '130k+'
  END AS salary_band,
  CASE
    WHEN lower(title) LIKE '%director%' OR lower(title) LIKE '%vp%' OR lower(title) LIKE '%head of%' THEN 'leadership'
    WHEN lower(title) LIKE '%senior%' OR lower(title) LIKE '%sr.%' OR lower(title) LIKE '%lead%' OR lower(title) LIKE '%principal%' THEN 'senior'
    WHEN lower(title) LIKE '%junior%' OR lower(title) LIKE '%jr.%' OR lower(title) LIKE '%entry%' OR lower(title) LIKE '%intern%' THEN 'junior'
    ELSE 'mid-level'
  END AS seniority_level,
  lower(description) LIKE '%remote%' AS is_remote,
  skills_arr, skills_count, posting_age_days,
  ingestion_date, created, source
FROM workspace.silver.job_postings_skills;

In [0]:
%sql
-- Skill demand per daily load (each ingestion day keeps its own rows — never replaced)
CREATE OR REPLACE TABLE workspace.gold.skill_demand AS
WITH per_day AS (
  SELECT ingestion_date, COUNT(*) AS day_total
  FROM workspace.silver.job_postings_skills
  GROUP BY ingestion_date
)
SELECT
  s.ingestion_date,
  skill,
  COUNT(DISTINCT job_id) AS job_count,
  ROUND(100.0 * COUNT(DISTINCT job_id) / MAX(d.day_total), 1) AS pct_of_postings,
  ROUND(AVG(salary_mid), 0) AS avg_salary_usd,
  MIN(salary_min) AS min_salary_usd,
  MAX(salary_max) AS max_salary_usd,
  COUNT(DISTINCT location_region) AS region_count
FROM (
  SELECT job_id, ingestion_date, explode(skills_arr) AS skill, salary_mid, salary_min, salary_max, location_region
  FROM workspace.silver.job_postings_skills
) s
JOIN per_day d ON d.ingestion_date = s.ingestion_date
GROUP BY s.ingestion_date, skill;

In [0]:
%sql
-- Hiring by location per daily load (history preserved by ingestion_date)
CREATE OR REPLACE TABLE workspace.gold.location_summary AS
SELECT
  ingestion_date, location_region, location_city,
  COUNT(*) AS job_count,
  ROUND(AVG(salary_mid), 0) AS avg_salary_usd,
  SUM(CASE WHEN skills_count > 0 THEN 1 ELSE 0 END) AS jobs_with_skills
FROM workspace.silver.job_postings_skills
GROUP BY ingestion_date, location_region, location_city;

In [0]:
%sql
SELECT * FROM workspace.gold.skill_demand
WHERE ingestion_date = (SELECT MAX(ingestion_date) FROM workspace.gold.skill_demand)
ORDER BY job_count DESC, avg_salary_usd DESC LIMIT 10;

In [0]:
%sql
SELECT country, COUNT(*) AS rows, SUM(CAST(is_remote AS INT)) AS remote_jobs, AVG(skills_count) AS avg_skills
FROM workspace.gold.job_postings
GROUP BY country;